# Post-Hoc Analysis of Sense-Annotated TSV Files

This notebook is dedicated to the analysis and comparison of TSV files produced by different LLM pipelines after sense assignment. Here, you will:

- Compute statistics such as how many tokens or MWEs were assigned `NEW_SENSE` by each model.
- Compare the sense assignment outputs of different models (e.g., ChatGPT, Gemini, Llama) to measure agreement and disagreement.

Use this notebook to gain insights into the quality and consistency of sense annotation across your models.

# Notebook Overview and Detailed Explanation

This notebook performs post-hoc analysis on sense-annotated TSV files. It is organized into the following sections:

1. **Introduction**: Explains the purpose of the notebook and the analysis objectives.
2. **Imports and Constants**: Loads required libraries and sets constants (e.g., columns for model agreement and flags for NEW_SENSE).
3. **Utility Functions**: 
   - *clean_data*: Removes trailing index markers and extra whitespace from annotation fields.
   - *get_token_agreement*: Compares two tokens for sense agreement and flags if a token was assigned a NEW_SENSE.
4. **MWE Functions**: 
   - *get_mwe_agreement*: Compares multi-word expressions (MWEs) by checking the agreement of their first tokens.
5. **Sentence and TSV Analysis**:
   - *get_sentence_agreement*: Compares sentences by evaluating both MWEs and standalone content tokens.
   - *get_tsv_files_agreement*: Processes TSV files to compute agreement statistics and outputs the results as CSV files.

In [23]:
import pandas as pd
import numpy as np
import os
import seaborn as sns
import matplotlib.pyplot as plt

from webanno_spacy_converter.parsers.tsv_parser_v3 import WebAnnoLEXISParser
from webanno_spacy_converter.models.annotation_token import AnnotationToken
from webanno_spacy_converter.models.sentence_with_mwes import AnnotatedSentenceWithMWEs, MultiWordExpression
from preprocessing import get_mwe_tokens, token_is_content_word, token_is_not_in_mwe
from config import SENSE_ID_FIELD, SENSE_ORIGIN, STATS_DIR

# --- Constants for Data Analysis ---
# Fields used to create the analysis DataFrame
# String used to denote that the model predicted NEW_SENSE
# In such cases, the first sense from the list was used and noted in the file
AGREEMENT = 'Agreement'         # Column for model agreement
ORIGIN_FIRST = "FIRST"          # Value indicating the first sense was used (model predicted NEW_SENSE)
FIRST_MODEL_NEW_SENSE = "First Model NEW_SENSE"  # Column for if first model predicted NEW_SENSE
SECOND_MODEL_NEW_SENSE = "Second Model NEW_SENSE"  # Column for if second model predicted NEW_SENSE

In [24]:
from config import L_LEMMA

def token_lemma_does_start_with_digit_or_is_missing(token):
    """
    Check if the token lemma starts with a digit
    :param token: Token to check
    :return: True if the token lemma starts with a digit, False otherwise
    """
    lemma = token.layers.get(L_LEMMA)
    if lemma is None:
        return True
    return lemma[0].isdigit()

def token_is_valid(token):
    """
    Check if the token is valid
    :param token: Token to check
    :return: True if the token is valid, False otherwise
    """
    return token_is_content_word(token) and token_is_not_in_mwe(token) and not token_lemma_does_start_with_digit_or_is_missing(token)


In [25]:
# Utility function to clean annotation fields (e.g., remove WebAnno index markers like [123])
def clean_data(field_text: str) -> str:
    """
    Remove trailing index markers (e.g., [123]) and extra whitespace from annotation fields.
    Args:
        field_text (str): The text to clean.
    Returns:
        str: Cleaned text without index markers.
    """
    return field_text.split("[")[0].strip()


def get_token_agreement(token1: AnnotationToken, token2: AnnotationToken) -> dict:
    """
    Compare two tokens and return a dictionary with agreement status and NEW_SENSE flags.
    Args:
        token1 (AnnotationToken): First token to compare.
        token2 (AnnotationToken): Second token to compare.
    Returns:
        dict: Agreement status and whether either model predicted NEW_SENSE (used first sense).
    """
    agreement = {
        'Agreement': False,
        'First Model NEW_SENSE': False,
        'Second Model NEW_SENSE': False
    }

    token1_sense = token1.layers.get(SENSE_ID_FIELD)
    token2_sense = token2.layers.get(SENSE_ID_FIELD)
    token1_origin = token1.layers.get(SENSE_ORIGIN)
    token2_origin = token2.layers.get(SENSE_ORIGIN)

    token1_sense = clean_data(token1_sense) if token1_sense else None
    token2_sense = clean_data(token2_sense) if token2_sense else None
    token1_origin = clean_data(token1_origin) if token1_origin else None
    token2_origin = clean_data(token2_origin) if token2_origin else None

    if not (token1_sense and token2_sense):
        print(f"Error: One of the tokens does not have a sense.")
        print(f"Token 1: {token1}")
        print(f"Token 1 Sense: {token1_sense}")
        print(f"Token 2: {token2}")
        print(f"Token 2 Sense: {token2_sense}")
        return agreement

    # Check if both tokens have the same sense assignment
    if token1_sense == token2_sense:
        agreement['Agreement'] = True
    agreement['First Model NEW_SENSE'] = token1_origin == ORIGIN_FIRST
    agreement['Second Model NEW_SENSE'] = token2_origin == ORIGIN_FIRST
    return agreement

In [26]:
def get_mwe_agreement(
    mwe1: MultiWordExpression, sentence1: AnnotatedSentenceWithMWEs,
    mwe2: MultiWordExpression, sentence2: AnnotatedSentenceWithMWEs
) -> dict:
    """
    Compare two MWEs and return a dictionary with agreement status and NEW_SENSE flags.
    Args:
        mwe1 (MultiWordExpression): First MWE to compare.
        sentence1 (AnnotatedSentenceWithMWEs): Sentence containing the first MWE.
        mwe2 (MultiWordExpression): Second MWE to compare.
        sentence2 (AnnotatedSentenceWithMWEs): Sentence containing the second MWE.
    Returns:
        dict: Agreement status and whether either model predicted NEW_SENSE (used first sense).
    Notes:
        Since sense values are repeated for all tokens in an MWE, we only need to check the first token of each MWE.
    """
    # Get the first token of each MWE (sense values are repeated, so one is enough)
    token1 = get_mwe_tokens(sentence1, mwe1)[0]
    token2 = get_mwe_tokens(sentence2, mwe2)[0]
    return get_token_agreement(token1, token2)

In [27]:
def get_sentence_agreement(sentence1: AnnotatedSentenceWithMWEs, sentence2: AnnotatedSentenceWithMWEs) -> tuple[list[dict], list[dict]]:
    """
    Compare two sentences and return a tuple of:
      - a list of dictionaries with agreement status and NEW_SENSE flags for each MWE and each content token not in an MWE
      - a list of dictionaries for those tokens and MWEs that were not aligned (disagreed)
    Args:
        sentence1 (AnnotatedSentenceWithMWEs): First sentence to compare.
        sentence2 (AnnotatedSentenceWithMWEs): Second sentence to compare.
    Returns:
        tuple: (agreements, not_aligned)
    """
    agreements = []
    not_aligned = []

    # First, compare MWEs (assumed to be aligned)
    for mwe1, mwe2 in zip(sentence1.mwes, sentence2.mwes):
        mwe_agreement = get_mwe_agreement(mwe1, sentence1, mwe2, sentence2)
        if not mwe_agreement['Agreement']:
            not_aligned.append({
                "Sentence": sentence1.text,
                "Token": clean_data(get_mwe_tokens(sentence1, mwe1)[0].text),
                FIRST_MODEL_NEW_SENSE: mwe_agreement['First Model NEW_SENSE'],
                SECOND_MODEL_NEW_SENSE: mwe_agreement['Second Model NEW_SENSE']
            })
        agreements.append(mwe_agreement)

    # Then, compare content tokens not in MWEs (assumed to be aligned)
    for token1, token2 in zip(sentence1.tokens, sentence2.tokens):
        if token_is_valid(token1) and token_is_valid(token2):
            token_agreement = get_token_agreement(token1, token2)
            if not token_agreement['Agreement']:
                not_aligned.append({
                    "Sentence": sentence1.text,
                    "Token": clean_data(token1.text),
                    "Sentence index": token1.sentence_index,
                    "Token index": token1.token_index,
                    FIRST_MODEL_NEW_SENSE: token_agreement['First Model NEW_SENSE'],
                    SECOND_MODEL_NEW_SENSE: token_agreement['Second Model NEW_SENSE']
                })
            agreements.append(token_agreement)
    return agreements, not_aligned

In [28]:
def get_tsv_files_agreement(
    tsv_file1: str,
    tsv_file2: str,
    output_dir: str,
    output_file: str,
    slice1: tuple[int, int] = None,
    slice2: tuple[int, int] = None
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Compare two TSV files and return DataFrames with agreement status and NEW_SENSE flags.

    Args:
        tsv_file1 (str): Path to the first TSV file.
        tsv_file2 (str): Path to the second TSV file.
        output_dir (str): Directory to save the output CSV files.
        output_file (str): Name of the output CSV file for agreements.
        slice1 (tuple[int, int], optional): Slice for the first TSV file (start, end). Defaults to None (use all).
        slice2 (tuple[int, int], optional): Slice for the second TSV file (start, end). Defaults to None (use all).

    Returns:
        tuple[pd.DataFrame, pd.DataFrame]:
            - DataFrame with agreement status for each token/MWE.
            - DataFrame with disagreement details (not aligned tokens/MWEs).
    """
    parser1 = WebAnnoLEXISParser(tsv_file1)
    parser2 = WebAnnoLEXISParser(tsv_file2)
    data = []
    disagreement_data = []
    sentences1 = parser1.parse()
    sentences2 = parser2.parse()

    # Apply slicing if provided
    if slice1 is not None:
        sentences1 = sentences1[slice1[0]:slice1[1]]
    if slice2 is not None:
        sentences2 = sentences2[slice2[0]:slice2[1]]

    # Compare sentences pairwise
    for sentence1, sentence2 in zip(sentences1, sentences2):
        if sentence1.text != sentence2.text:
            print(f"Warning: Sentences do not match:\n{sentence1.text}\n{sentence2.text}")
            continue
        agreements, not_aligned = get_sentence_agreement(sentence1, sentence2)
        data.extend(agreements)
        disagreement_data.extend(not_aligned)

    # Create DataFrames for agreement and disagreement
    df_agreement = pd.DataFrame(data)
    df_disagreement = pd.DataFrame(disagreement_data)

    # Ensure output directory exists
    os.makedirs(output_dir, exist_ok=True)

    # Save results to CSV files (tab-separated)
    df_agreement.to_csv(os.path.join(output_dir, output_file), index=False, sep="\t")
    df_disagreement.to_csv(os.path.join(output_dir, f"disagreement_{output_file}"), index=False, sep="\t")

    return df_agreement, df_disagreement

In [29]:
from config import OUTPUT_DIR, STATS_DIR
from pathlib import Path
import os


def compute_file_info(base_file_name: str, start_sentence: str, end_sentence: str, model1: str, model2: str, input_dir: Path) -> tuple:
    """
    Compute and print file information based on the provided parameters.
    
    Args:
        base_file_name: The base name for the files.
        start_sentence: Starting sentence identifier.
        end_sentence: Ending sentence identifier.
        model1: Name of the first model.
        model2: Name of the second model.
        input_dir: Directory path where models have placed results.
        
    Returns:
        A tuple (file_path1, file_path2) representing the paths for the two TSV files.
    """
    filename1 = f"{base_file_name}_{start_sentence}_{end_sentence}_{model1}.tsv"
    filename2 = f"{base_file_name}_{start_sentence}_{end_sentence}_{model2}.tsv"
    file_path1 = input_dir / filename1
    file_path2 = input_dir / filename2
    print(f"Comparing {filename1} and {filename2}")
    print(f"File 1: {file_path1}")
    print(f"File 2: {file_path2}")
    if not os.path.exists(file_path1):
        print(f"File {file_path1} does not exist.")
    if not os.path.exists(file_path2):
        print(f"File {file_path2} does not exist.")
    return file_path1, file_path2


model1 = "ChatGPT-3-5"
model2 = "ChatGPT-4-1"
start_sentence = "0001"
end_sentence = "0500"

base_file_name = "LexiSense_Inception_test"

file_path1, file_path2 = compute_file_info(base_file_name, start_sentence, end_sentence, model1, model2, OUTPUT_DIR)

Comparing LexiSense_Inception_test_0001_0500_ChatGPT-3-5.tsv and LexiSense_Inception_test_0001_0500_ChatGPT-4-1.tsv
File 1: e:\Github\LexiSense-SR\output\LexiSense_Inception_test_0001_0500_ChatGPT-3-5.tsv
File 2: e:\Github\LexiSense-SR\output\LexiSense_Inception_test_0001_0500_ChatGPT-4-1.tsv


In [30]:
aggreement_df, disagreement_df = get_tsv_files_agreement(
    file_path1,
    file_path2,
    STATS_DIR,
    f"agreement_{model1}_{model2}_{start_sentence}_{end_sentence}.csv")



In [31]:
def calculate_alignment_percentage(df: pd.DataFrame) -> float:
    """
    Calculate the percentage of aligned entries in the DataFrame.
    Alignment is determined by the 'Agreement' column.
    
    Args:
        df (pd.DataFrame): DataFrame containing alignment results with a boolean 'Agreement' column.
    
    Returns:
        float: Percentage of True values in the 'Agreement' column.
    """
    if df.empty:
        return 0.0
    aligned_count = df['Agreement'].sum()
    total_count = len(df)
    return (aligned_count / total_count) * 100

# Example usage:
# percent = calculate_alignment_percentage(aggreement_df)
# print(f"Alignment Percentage: {percent:.2f}%")

In [32]:
percent = calculate_alignment_percentage(aggreement_df)
print(f"Alignment Percentage: {percent:.2f}%")

Alignment Percentage: 78.88%


In [33]:
def calculate_new_sense_agreement_percentage(df: pd.DataFrame) -> float:
    """
    Calculate the percentage of rows where both models predicted NEW_SENSE.
    This is determined by both 'First Model NEW_SENSE' and 'Second Model NEW_SENSE' being True.
    
    Args:
        df (pd.DataFrame): DataFrame containing alignment results.
    
    Returns:
        float: Percentage where both models agreed on NEW_SENSE.
    """
    if df.empty:
        return 0.0
    new_sense_count = df[(df['First Model NEW_SENSE'] == True) & (df['Second Model NEW_SENSE'] == True)].shape[0]
    total_count = len(df)
    return (new_sense_count / total_count) * 100

In [34]:
new_sense_pct = calculate_new_sense_agreement_percentage(aggreement_df)
print(f"New Sense Agreement Percentage: {new_sense_pct:.2f}%")

New Sense Agreement Percentage: 4.00%


In [35]:
def calculate_new_sense_statistics(df: pd.DataFrame) -> dict:
    """
    Compute detailed statistics for NEW_SENSE predictions in the alignment DataFrame.

    Returns a dictionary with:
      - total: Total number of rows.
      - both_agree: Count where both models predicted NEW_SENSE.
      - first_only: Count where only the first model predicted NEW_SENSE.
      - second_only: Count where only the second model predicted NEW_SENSE.
      - only_one: Count where exactly one model predicted NEW_SENSE.
      - at_least_one: Count where at least one model predicted NEW_SENSE.
      - none: Count where neither model predicted NEW_SENSE.
      - both_pct: Percentage with both models NEW_SENSE.
      - first_only_pct: Percentage with only the first model NEW_SENSE.
      - second_only_pct: Percentage with only the second model NEW_SENSE.
      - only_one_pct: Percentage with exactly one model NEW_SENSE.
      - at_least_one_pct: Percentage with at least one model NEW_SENSE.
      - none_pct: Percentage with no NEW_SENSE.
    """
    if df.empty:
        return {}
    
    total = len(df)
    both_agree = df[(df['First Model NEW_SENSE'] == True) & (df['Second Model NEW_SENSE'] == True)].shape[0]
    first_only = df[(df['First Model NEW_SENSE'] == True) & (df['Second Model NEW_SENSE'] == False)].shape[0]
    second_only = df[(df['First Model NEW_SENSE'] == False) & (df['Second Model NEW_SENSE'] == True)].shape[0]
    only_one = first_only + second_only  # or use XOR
    at_least_one = df[(df['First Model NEW_SENSE'] == True) | (df['Second Model NEW_SENSE'] == True)].shape[0]
    none = df[(df['First Model NEW_SENSE'] == False) & (df['Second Model NEW_SENSE'] == False)].shape[0]
    
    both_pct = (both_agree / total) * 100
    first_only_pct = (first_only / total) * 100
    second_only_pct = (second_only / total) * 100
    only_one_pct = (only_one / total) * 100
    at_least_one_pct = (at_least_one / total) * 100
    none_pct = (none / total) * 100

    return {
        "total": total,
        "both_agree": both_agree,
        "first_only": first_only,
        "second_only": second_only,
        "only_one": only_one,
        "at_least_one": at_least_one,
        "none": none,
        "both_pct": both_pct,
        "first_only_pct": first_only_pct,
        "second_only_pct": second_only_pct,
        "only_one_pct": only_one_pct,
        "at_least_one_pct": at_least_one_pct,
        "none_pct": none_pct
    }

In [36]:
def compute_and_print_new_sense_statistics(df: pd.DataFrame, model1: str, model2: str, start_sentence: str, end_sentence: str) -> None:
    """
    Compute the NEW_SENSE statistics from the DataFrame and print them in a formatted manner.
    
    Args:
        df: DataFrame containing NEW_SENSE columns.
        model1: First model name.
        model2: Second model name.
        start_sentence: Start sentence identifier.
        end_sentence: End sentence identifier.
    """
    stats = calculate_new_sense_statistics(df)
    alignment_pct = calculate_alignment_percentage(df)
    
    print("=== NEW_SENSE Statistics ===")
    print(f"Model 1              : {model1}")
    print(f"Model 2              : {model2}")
    print(f"Sentence Range       : {start_sentence} - {end_sentence}")
    print(f"Total Rows           : {stats.get('total', 0)}")
    print(f"Alignment Percentage : {alignment_pct:.2f}%")
    print("New Sense Breakdown:")
    print(f"  Both agree         : {stats.get('both_agree', 0)} ({stats.get('both_pct', 0):.2f}%)")
    print(f"  Only {model1}       : {stats.get('first_only', 0)} ({stats.get('first_only_pct', 0):.2f}%)")
    print(f"  Only {model2}       : {stats.get('second_only', 0)} ({stats.get('second_only_pct', 0):.2f}%)")
    print(f"  Only one (total)   : {stats.get('only_one', 0)} ({stats.get('only_one_pct', 0):.2f}%)")
    print(f"  At least one       : {stats.get('at_least_one', 0)} ({stats.get('at_least_one_pct', 0):.2f}%)")
    print(f"  None               : {stats.get('none', 0)} ({stats.get('none_pct', 0):.2f}%)")

# Example usage:
compute_and_print_new_sense_statistics(aggreement_df, model1, model2, start_sentence, end_sentence)

=== NEW_SENSE Statistics ===
Model 1              : ChatGPT-3-5
Model 2              : ChatGPT-4-1
Sentence Range       : 0001 - 0500
Total Rows           : 3703
Alignment Percentage : 78.88%
New Sense Breakdown:
  Both agree         : 148 (4.00%)
  Only ChatGPT-3-5       : 231 (6.24%)
  Only ChatGPT-4-1       : 50 (1.35%)
  Only one (total)   : 281 (7.59%)
  At least one       : 429 (11.59%)
  None               : 3274 (88.41%)


In [37]:
start_sentence = "0501"
end_sentence = "1000"

file_path1, file_path2 = compute_file_info(base_file_name, start_sentence, end_sentence, model1, model2, OUTPUT_DIR)
aggreement_df, disagreement_df = get_tsv_files_agreement(
    file_path1,
    file_path2,
    STATS_DIR,
    f"agreement_{model1}_{model2}_{start_sentence}_{end_sentence}.csv")
percent = calculate_alignment_percentage(aggreement_df)
print(f"Alignment Percentage: {percent:.2f}%")


Comparing LexiSense_Inception_test_0501_1000_ChatGPT-3-5.tsv and LexiSense_Inception_test_0501_1000_ChatGPT-4-1.tsv
File 1: e:\Github\LexiSense-SR\output\LexiSense_Inception_test_0501_1000_ChatGPT-3-5.tsv
File 2: e:\Github\LexiSense-SR\output\LexiSense_Inception_test_0501_1000_ChatGPT-4-1.tsv
Alignment Percentage: 77.03%


In [38]:
compute_and_print_new_sense_statistics(aggreement_df, model1, model2, start_sentence, end_sentence)


=== NEW_SENSE Statistics ===
Model 1              : ChatGPT-3-5
Model 2              : ChatGPT-4-1
Sentence Range       : 0501 - 1000
Total Rows           : 3613
Alignment Percentage : 77.03%
New Sense Breakdown:
  Both agree         : 123 (3.40%)
  Only ChatGPT-3-5       : 268 (7.42%)
  Only ChatGPT-4-1       : 46 (1.27%)
  Only one (total)   : 314 (8.69%)
  At least one       : 437 (12.10%)
  None               : 3176 (87.90%)


In [39]:
start_sentence = "1001"
end_sentence = "1500"
file_path1, file_path2 = compute_file_info(base_file_name, start_sentence, end_sentence, model1, model2, OUTPUT_DIR)
aggreement_df, disagreement_df = get_tsv_files_agreement(
    file_path1,
    file_path2,
    STATS_DIR,
    f"agreement_{model1}_{model2}_{start_sentence}_{end_sentence}.csv")
percent = calculate_alignment_percentage(aggreement_df)
print(f"Alignment Percentage: {percent:.2f}%")

Comparing LexiSense_Inception_test_1001_1500_ChatGPT-3-5.tsv and LexiSense_Inception_test_1001_1500_ChatGPT-4-1.tsv
File 1: e:\Github\LexiSense-SR\output\LexiSense_Inception_test_1001_1500_ChatGPT-3-5.tsv
File 2: e:\Github\LexiSense-SR\output\LexiSense_Inception_test_1001_1500_ChatGPT-4-1.tsv
Alignment Percentage: 77.79%


In [41]:
compute_and_print_new_sense_statistics(aggreement_df, model1, model2, start_sentence, end_sentence)

=== NEW_SENSE Statistics ===
Model 1              : ChatGPT-3-5
Model 2              : ChatGPT-4-1
Sentence Range       : 1001 - 1500
Total Rows           : 3426
Alignment Percentage : 77.79%
New Sense Breakdown:
  Both agree         : 132 (3.85%)
  Only ChatGPT-3-5       : 248 (7.24%)
  Only ChatGPT-4-1       : 35 (1.02%)
  Only one (total)   : 283 (8.26%)
  At least one       : 415 (12.11%)
  None               : 3011 (87.89%)


In [44]:
start_sentence = "1501"
end_sentence = "2000"
file_path1, file_path2 = compute_file_info(base_file_name, start_sentence, end_sentence, model1, model2, OUTPUT_DIR)
aggreement_df, disagreement_df = get_tsv_files_agreement(
    file_path1,
    file_path2,
    STATS_DIR,
    f"agreement_{model1}_{model2}_{start_sentence}_{end_sentence}.csv")


Comparing LexiSense_Inception_test_1501_2000_ChatGPT-3-5.tsv and LexiSense_Inception_test_1501_2000_ChatGPT-4-1.tsv
File 1: e:\Github\LexiSense-SR\output\LexiSense_Inception_test_1501_2000_ChatGPT-3-5.tsv
File 2: e:\Github\LexiSense-SR\output\LexiSense_Inception_test_1501_2000_ChatGPT-4-1.tsv


In [43]:
compute_and_print_new_sense_statistics(aggreement_df, model1, model2, start_sentence, end_sentence)

=== NEW_SENSE Statistics ===
Model 1              : ChatGPT-3-5
Model 2              : ChatGPT-4-1
Sentence Range       : 1501 - 2000
Total Rows           : 3437
Alignment Percentage : 77.42%
New Sense Breakdown:
  Both agree         : 119 (3.46%)
  Only ChatGPT-3-5       : 266 (7.74%)
  Only ChatGPT-4-1       : 39 (1.13%)
  Only one (total)   : 305 (8.87%)
  At least one       : 424 (12.34%)
  None               : 3013 (87.66%)


In [45]:
start_sentence = "2001"
end_sentence = "2025"
file_path1, file_path2 = compute_file_info(base_file_name, start_sentence, end_sentence, model1, model2, OUTPUT_DIR)
aggreement_df, disagreement_df = get_tsv_files_agreement(
    file_path1,
    file_path2,
    STATS_DIR,
    f"agreement_{model1}_{model2}_{start_sentence}_{end_sentence}.csv")

Comparing LexiSense_Inception_test_2001_2025_ChatGPT-3-5.tsv and LexiSense_Inception_test_2001_2025_ChatGPT-4-1.tsv
File 1: e:\Github\LexiSense-SR\output\LexiSense_Inception_test_2001_2025_ChatGPT-3-5.tsv
File 2: e:\Github\LexiSense-SR\output\LexiSense_Inception_test_2001_2025_ChatGPT-4-1.tsv


In [46]:
compute_and_print_new_sense_statistics(aggreement_df, model1, model2, start_sentence, end_sentence)

=== NEW_SENSE Statistics ===
Model 1              : ChatGPT-3-5
Model 2              : ChatGPT-4-1
Sentence Range       : 2001 - 2025
Total Rows           : 188
Alignment Percentage : 73.94%
New Sense Breakdown:
  Both agree         : 5 (2.66%)
  Only ChatGPT-3-5       : 14 (7.45%)
  Only ChatGPT-4-1       : 2 (1.06%)
  Only one (total)   : 16 (8.51%)
  At least one       : 21 (11.17%)
  None               : 167 (88.83%)
